In [1]:
import os
from os.path import join, isfile, isdir
from urllib.request import urlretrieve
from anndata import read_h5ad
import anndata as ad

from vitessce import (
    VitessceConfig,
    Component as cm,
    CoordinationType as ct,
    AnnDataWrapper,
)
from vitessce.data_utils import (
    optimize_adata,
    VAR_CHUNK_SIZE,
)

/opt/conda/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


In [2]:
print(ad.__version__)

0.10.9


In [2]:
import zarr

print(zarr.__version__)

2.18.7


In [3]:
adata_filepath = join("data", "habib17.processed.h5ad")
if not isfile(adata_filepath):
    os.makedirs("data", exist_ok=True)
    urlretrieve('https://covid19.cog.sanger.ac.uk/habib17.processed.h5ad', adata_filepath)

In [4]:
adata = read_h5ad(adata_filepath)

/opt/conda/lib/python3.11/site-packages/anndata/compat/__init__.py:329: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/opt/conda/lib/python3.11/site-packages/anndata/compat/__init__.py:329: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


In [5]:
top_dispersion = adata.var["dispersions_norm"][
    sorted(
        range(len(adata.var["dispersions_norm"])),
        key=lambda k: adata.var["dispersions_norm"][k],
    )[-51:][0]
]
adata.var["top_highly_variable"] = (
    adata.var["dispersions_norm"] > top_dispersion
)

/tmp/ipykernel_2304/862524975.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  key=lambda k: adata.var["dispersions_norm"][k],
/tmp/ipykernel_2304/862524975.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  top_dispersion = adata.var["dispersions_norm"][


In [6]:
adata.obs

,CellType,n_counts,log1p_n_counts,n_genes,log1p_n_genes,percent_mito,percent_ribo,percent_hb,percent_top50
index,,,,,,,,,
hHP1_ACTCAATAGCAA-habib17,exCA1,2393.0,7.780721,1487,7.305188,0.376097,1.295445,0.000000,26.619306
hHP1_TTCCCGTTAAAG-habib17,exCA3,1963.0,7.582738,1325,7.189922,0.101885,0.458482,0.101885,25.624045
hHP1_GTCATTGAATCA-habib17,ASC1,1829.0,7.512071,931,6.837333,0.109349,0.656096,0.000000,44.942592
hHP1_CACCTTCAATAC-habib17,exCA1,2597.0,7.862497,1711,7.445418,0.231036,1.463227,0.000000,19.060454
hHP1_ATACATGTTGTC-habib17,exCA3,1890.0,7.544861,1288,7.161622,0.211640,0.846561,0.000000,23.227513
...,...,...,...,...,...,...,...,...,...
PFC.CD_CAACCAATTTCG-habib17,Unclassified,481.0,6.177944,416,6.033086,0.831601,0.831601,0.000000,23.700624
PFC.CD_CACGCTCCCCTA-habib17,exPFC1,271.0,5.605802,217,5.384495,0.000000,0.738007,0.000000,38.376384
PFC.CD_GCTCTACAACCG-habib17,END,522.0,6.259582,429,6.063785,2.107280,2.107280,0.000000,27.394636


In [7]:
zarr_filepath = join("data", "habib17.processed.zarr")
if not isdir(zarr_filepath):
    adata = optimize_adata(
        adata,
        obs_cols=["CellType"],
        obsm_keys=["X_umap"],
        optimize_X=True,
        var_cols=["top_highly_variable"],
    )
    adata.write_zarr(zarr_filepath, chunks=[adata.shape[0], VAR_CHUNK_SIZE])

In [8]:
zarr.consolidate_metadata(zarr_filepath)

<zarr.hierarchy.Group '/'>

In [9]:
print(zarr_filepath)

data/habib17.processed.zarr


In [10]:
vc = VitessceConfig(schema_version="1.0.15", name='Habib et al', description='COVID-19 Healthy Donor Brain')


In [11]:
dataset = vc.add_dataset(name='Brain').add_object(AnnDataWrapper(
        adata_path=zarr_filepath,
        obs_embedding_paths=["obsm/X_umap"],
        obs_embedding_names=["UMAP"],
        obs_set_paths=["obs/CellType"],
        obs_set_names=["Cell Type"],
        obs_feature_matrix_path="X",
        initial_feature_filter_path="var/top_highly_variable"
    )
)

In [12]:
scatterplot = vc.add_view(cm.SCATTERPLOT, dataset=dataset, mapping="UMAP")
cell_sets = vc.add_view(cm.OBS_SETS, dataset=dataset)
genes = vc.add_view(cm.FEATURE_LIST, dataset=dataset)
heatmap = vc.add_view(cm.HEATMAP, dataset=dataset)

In [13]:
vc.layout((scatterplot | cell_sets) / (heatmap | genes));


In [14]:
vw = vc.widget()
vw